# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os




sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-4
num_classes = 50

notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

## **1. Preprocessing**

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [4]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [5]:
heavy_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="heavy",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

light_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="light",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

val_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="none",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)


train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE, train_preprocessor_light=light_training_preprocessor, train_preprocessor_heavy=heavy_training_preprocessor, val_preprocessor=val_preprocessor)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


## **2. Modèle**

In [6]:
from torch.optim.lr_scheduler import CosineAnnealingLR
def create_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

## **3. F1-score**

In [7]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [8]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [9]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## **5. Entrainement**

**Vérif des labels**

In [10]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

**Entrainement**

In [11]:
import csv
best_f1 = 0.0
best_model_path = "best_model.pth"

# Configuration du logging CSV
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro']

# Initialisation du fichier (écrase le précédent s'il existe)
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()

    scheduler.step()
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

100%|██████████| 201/201 [01:53<00:00,  1.76it/s]


Epoch 1/25
Train Loss: 3.2455 | Train Acc: 0.2835
Train F1 Macro: 0.2732
Train F1 per class: [np.float64(0.4862745098039215), np.float64(0.6772908366533866), np.float64(0.10152284263959391), np.float64(0.259927797833935), np.float64(0.13526570048309178), np.float64(0.12413793103448276), np.float64(0.16363636363636364), np.float64(0.2780748663101604), np.float64(0.5469387755102042), np.float64(0.41379310344827586), np.float64(0.42685851318944845), np.float64(0.1714285714285714), np.float64(0.11494252873563218), np.float64(0.35), np.float64(0.06282722513089005), np.float64(0.1802816901408451), np.float64(0.2183406113537118), np.float64(0.1832460732984293), np.float64(0.20731707317073172), np.float64(0.3370786516853933), np.float64(0.4132231404958677), np.float64(0.19534883720930232), np.float64(0.16981132075471697), np.float64(0.473469387755102), np.float64(0.1764705882352941), np.float64(0.1793103448275862), np.float64(0.578397212543554), np.float64(0.32786885245901637), np.float64(0.0

Val   Loss: 2.5124
Val   F1 Macro: 0.2493
Val   F1 per class: [0, np.float64(0.2857142857142857), np.float64(0.21311475409836067), np.float64(0.5), 0, np.float64(0.15053763440860218), np.float64(0.3333333333333333), 0, np.float64(0.4), np.float64(0.6666666666666665), np.float64(0.2857142857142857), np.float64(0.25), np.float64(0.21794871794871795), 0, 0, np.float64(0.2), 0, np.float64(0.2288557213930348), np.float64(0.3667820069204153), np.float64(0.7712418300653594), np.float64(0.6666666666666666), 0, np.float64(0.5240641711229946), np.float64(0.7142857142857143), np.float64(0.3557312252964427), np.float64(0.5448504983388704), np.float64(0.5), 0, np.float64(0.2676056338028169), 0, np.float64(0.5), 0, np.float64(0.10256410256410256), 0, np.float64(0.33333333333333337), 0, np.float64(0.29069767441860467), 0, 0, 0, np.float64(0.6551724137931034), np.float64(0.24137931034482765), 0, np.float64(0.07874015748031496), np.float64(0.5454545454545454), 0, 0, 0, np.float64(0.6666666666666666), n

100%|██████████| 201/201 [02:09<00:00,  1.55it/s]


Epoch 2/25
Train Loss: 1.7298 | Train Acc: 0.6246
Train F1 Macro: 0.6018
Train F1 per class: [np.float64(0.9078947368421052), np.float64(0.910828025477707), np.float64(0.0909090909090909), np.float64(0.7064220183486237), np.float64(0.5775401069518716), np.float64(0.44609665427509293), np.float64(0.5575221238938054), np.float64(0.7553956834532374), np.float64(0.8729641693811074), np.float64(0.6042553191489363), np.float64(0.7596899224806201), np.float64(0.6920152091254752), np.float64(0.2962962962962963), np.float64(0.8571428571428572), np.float64(0.451063829787234), np.float64(0.550943396226415), np.float64(0.6688524590163935), np.float64(0.4423676012461059), np.float64(0.359447004608295), np.float64(0.6446280991735537), np.float64(0.861842105263158), np.float64(0.5064377682403434), np.float64(0.5988372093023256), np.float64(0.6996466431095406), np.float64(0.2711864406779661), np.float64(0.4365781710914454), np.float64(0.9496402877697843), np.float64(0.7601246105919003), np.float64(0.

...


Epoch 24/25
Train Loss: 0.1207 | Train Acc: 0.9663
Train F1 Macro: 0.9666
Train F1 per class: [np.float64(1.0), np.float64(0.9962546816479401), np.float64(0.8957528957528959), np.float64(1.0), np.float64(0.9956331877729258), np.float64(0.8976377952755906), np.float64(1.0), np.float64(0.9925925925925926), np.float64(0.9959514170040485), np.float64(1.0), np.float64(1.0), np.float64(0.9927007299270074), np.float64(0.889763779527559), np.float64(1.0), np.float64(0.971223021582734), np.float64(0.9927007299270074), np.float64(0.9965397923875432), np.float64(0.9583333333333333), np.float64(0.900763358778626), np.float64(0.9575289575289576), np.float64(1.0), np.float64(0.9919354838709677), np.float64(0.9051724137931035), np.float64(0.9871244635193133), np.float64(0.9125475285171103), np.float64(0.9272030651340997), np.float64(1.0), np.float64(1.0), np.float64(0.9212598425196851), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.8825910931174089), np.float64(1.0), np.float64(1.0

Val   Loss: 0.7105
Val   F1 Macro: 0.6136
Val   F1 per class: [0, 0, np.float64(0.7009345794392524), np.float64(1.0), np.float64(0.8), np.float64(0.7976190476190477), np.float64(0.6666666666666666), np.float64(1.0), np.float64(0.6666666666666666), np.float64(0.888888888888889), np.float64(0.6666666666666666), np.float64(0.8571428571428571), np.float64(0.7388535031847134), 0, np.float64(0.7777777777777778), np.float64(0.6666666666666666), 0, np.float64(0.7222222222222223), np.float64(0.7857142857142857), np.float64(0.9420289855072463), np.float64(1.0), np.float64(0.7692307692307692), np.float64(0.8305084745762713), np.float64(0.896551724137931), np.float64(0.8494208494208494), np.float64(0.8965517241379309), np.float64(1.0), 0, np.float64(0.7796610169491526), 0, 0, np.float64(0.6666666666666666), np.float64(0.6412213740458016), 0, np.float64(1.0), 0, np.float64(0.7216494845360825), np.float64(0.7142857142857143), 0, np.float64(0.823529411764706), np.float64(0.8702290076335878), np.float

100%|██████████| 201/201 [01:54<00:00,  1.76it/s]


Epoch 25/25
Train Loss: 0.1100 | Train Acc: 0.9718
Train F1 Macro: 0.9715
Train F1 per class: [np.float64(0.9959839357429718), np.float64(0.9959514170040485), np.float64(0.9176470588235294), np.float64(1.0), np.float64(0.9890909090909091), np.float64(0.9105691056910571), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.9960474308300395), np.float64(0.9929577464788732), np.float64(0.9128630705394191), np.float64(1.0), np.float64(0.968503937007874), np.float64(0.9767441860465117), np.float64(0.9924812030075187), np.float64(0.9578544061302682), np.float64(0.9393939393939394), np.float64(0.9568627450980391), np.float64(0.9961389961389961), np.float64(0.9846153846153846), np.float64(0.9068825910931174), np.float64(0.9961389961389961), np.float64(0.8659003831417624), np.float64(0.9491525423728814), np.float64(0.9954751131221719), np.float64(0.9959183673469388), np.float64(0.92), np.float64(0.9929577464788732), np.float64(1.0), np.float64(1.0), np.float64(0.85

Val   Loss: 0.7110
Val   F1 Macro: 0.6159
Val   F1 per class: [0, 0, np.float64(0.6862745098039216), np.float64(1.0), np.float64(0.8), np.float64(0.7730061349693251), np.float64(0.6666666666666666), np.float64(1.0), np.float64(0.6666666666666666), np.float64(0.888888888888889), np.float64(0.6666666666666666), np.float64(0.8571428571428571), np.float64(0.7388535031847134), 0, np.float64(0.742857142857143), np.float64(0.6666666666666666), 0, np.float64(0.7012987012987013), np.float64(0.7958115183246073), np.float64(0.9361702127659575), np.float64(1.0), np.float64(0.7692307692307692), np.float64(0.8407079646017699), np.float64(0.912280701754386), np.float64(0.8372093023255813), np.float64(0.9), np.float64(1.0), 0, np.float64(0.7863247863247863), 0, 0, np.float64(0.6666666666666666), np.float64(0.6356589147286822), 0, np.float64(1.0), 0, np.float64(0.6938775510204083), np.float64(0.6666666666666666), 0, np.float64(0.875), np.float64(0.8921933085501859), np.float64(0.625), np.float64(0.4444

## **6. Création du fichier submission**

In [12]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [13]:
preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    resize_method="pad",
    target_size=(224, 224),
)


test_dataset = BeeDataset(train=False, transform=preprocessor)


submission(model, batch_size=32, transform=preprocessor, save_path="submission.csv")

/tmp/ipykernel_6271/2135808320.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))


Submission saved to submission.csv
